## 提取中证500成分股的因子数据 — 教程

### 整体流程说明

本 Notebook 的目标是从聚宽（JoinQuant）平台提取 **中证500指数成分股** 的 **53个量化因子** 数据，最终输出为 CSV 文件。

---

### 第一步：获取成分股列表

中证500指数的成分股**不是固定不变的**，每半年会调整一次。因此我们需要：
- 遍历每一个交易日（2007-01-15 ~ 2026-05-16，共约4696天）
- 查询该日中证500包含哪些股票（每天约500只）
- 汇总历史上所有曾属于中证500的股票（共约1778只）

---

### 第二步：定义要提取的因子

待提取的53个因子，涵盖基本面、技术面、估值、质量、波动/风险等几大类。具体列表见下方代码单元格。

---

### 第三步：分窗口提取因子

由于聚宽 API 有**单次请求上限**（20万个数据单元格），无法一次性获取所有数据。

数据量 = 股票数 × 因子数 × 交易日数，总量约 1.24 亿，远超上限。

解决方案：把时间范围**拆成多个小窗口**，每个窗口内：
1. 调用 `get_factor_values` 获取因子数据
2. 把返回的宽表转为长表格式
3. **过滤**：只保留当天真正属于中证500的股票（因为窗口内不同日期的成分股可能不同）
4. 转换股票代码格式并排序，追加写入 CSV

---

### 第四步：预览输出 CSV

最终输出格式如下（55列 = 日期 + 股票代码 + 53个因子）：

```csv
date,symbol,factor_1,factor_2,...,factor_53
2007-01-15,sh600000,10.5,0.085,...,1.2
2007-01-15,sh600039,-1.8,0.047,...,0.9
```

In [ ]:
# ============================================================
# 导入所需的库和配置参数
# ============================================================

import gc        # 垃圾回收模块，用于手动释放内存，防止处理大量数据时内存溢出
import os        # 操作系统接口，用于检查文件是否存在、删除文件等
import time      # 时间模块，用于记录各个步骤的耗时，方便定位性能瓶颈
import pandas as pd  # 数据分析库，提供 DataFrame 数据结构，是整个数据处理的核心工具

# 聚宽（JoinQuant）研究环境的特殊处理：
# 聚宽平台会把 get_index_stocks、get_trade_days 等 jqdata API 直接注入到全局命名空间中。
# 但有些环境不支持 `from jqdata import get_index_stocks` 这种显式导入方式，
# 所以这里用 try/except 做兼容：能导入就导入，不能导入就跳过（假设已经在全局命名空间中了）。
try:
    from jqdata import *
except ImportError:
    pass

# 从 jqfactor 库中导入 get_factor_values 函数
# 这个函数是核心：给定股票列表和因子名称，它能批量获取这些股票在指定时间段内的因子数值
from jqfactor import get_factor_values

# 尝试导入 tqdm 进度条库，用于显示循环处理进度
# 如果环境没有安装 tqdm，就把它设为 None，后面代码会据此判断是否显示进度条
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

# ============================================================
# 全局参数配置
# ============================================================

# 中证500指数的代码，XSHG 表示上海证券交易所
# 中证500是由A股中市值排名301-800的500只股票组成的指数，代表中盘股表现
INDEX_CODE = '000905.XSHG'

# 数据起始日期：中证500指数于2007年1月15日正式发布，所以从这天开始获取数据
START_DATE = '2007-01-15'

# 数据结束日期：获取到2026年5月16日为止的数据
END_DATE = '2026-05-16'

# 最终输出的 CSV 文件名
OUTPUT_CSV = 'factor_0516.csv'

# 聚宽 API 的单次请求上限：一次 get_factor_values 调用最多处理 200000 个数据单元格
# 一个数据单元格 = 1只股票 × 1个因子 × 1个交易日
# 这里设为 180000 留出安全余量，避免触发 API 限制导致请求失败
MAX_REQUEST_CELLS = 180000

# ============================================================
# 待提取的因子列表
# ============================================================
# 这些因子来自聚宽的 jqfactor 库，涵盖以下几个大类：
#
# 【基本面因子】（来自财务报表）
#   - 资产减值损失、现金流价格比、EBIT（息税前利润）、营运资金等
#
# 【技术面因子】（来自行情数据计算）
#   - AR（人气指标）、ARBR（人气意愿指标）、ATR6（6日平均真实波幅）等
#
# 【估值因子】
#   - 账面市值比、现金收益价格比、盈利收益率、成长性等
#
# 【质量因子】
#   - 超速动比率、资产负债率、经营现金流覆盖倍数等
#
# 【波动/风险因子】
#   - 峰度(Kurtosis)、偏度(Skewness)、夏普比率(sharpe_ratio)、Beta等
#
jqfactors_list = [
    'asset_impairment_loss_ttm',           # 资产减值损失（TTM，即滚动12个月）
    'cash_flow_to_price_ratio',            # 现金流价格比：经营现金流 / 总市值
    'EBIT',                                # 息税前利润
    'net_working_capital',                 # 净营运资金 = 流动资产 - 流动负债
    'non_recurring_gain_loss',             # 非经常性损益
    'sales_to_price_ratio',                # 营收价格比：营业收入 / 总市值
    'AR',                                  # 人气指标（技术指标）
    'ARBR',                                # 人气意愿指标（技术指标）
    'ATR6',                                # 6日平均真实波幅（衡量波动性）
    'DAVOL10',                             # 10日平均换手率偏差
    'MAWVAD',                              # WVAD（威廉变异离散量）的移动平均
    'BBIC',                                # 布林带指标
    'VOL10',                               # 10日成交量
    'VDIFF',                               # 成交量差值
    'VEMA26',                              # 成交量26日指数移动平均
    'VMACD',                               # 成交量MACD指标
    'VR',                                  # 容量比率（成交量比率）
    'single_day_VPT',                      # 单日VPT（量价趋势指标）
    'single_day_VPT_12',                   # 12日VPT
    'capital_reserve_fund_per_share',      # 每股资本公积
    'net_operate_cash_flow_per_share',     # 每股经营现金流
    'operating_profit_per_share',          # 每股营业利润
    'total_operating_revenue_per_share',   # 每股营业总收入
    'surplus_reserve_fund_per_share',      # 每股盈余公积
    'ACCA',                                # 应计利润（衡量盈利质量）
    'account_receivable_turnover_days',    # 应收账款周转天数
    'account_receivable_turnover_rate',    # 应收账款周转率
    'adjusted_profit_to_total_profit',     # 调整后利润占利润总额比
    'super_quick_ratio',                   # 超速动比率（流动性指标）
    'MLEV',                                # 主杠杆率
    'debt_to_equity_ratio',                # 产权比率（负债/所有者权益）
    'debt_to_tangible_equity_ratio',       # 有形资产负债率
    'equity_to_fixed_asset_ratio',         # 股东权益与固定资产比
    'fixed_asset_ratio',                   # 固定资产占比
    'intangible_asset_ratio',              # 无形资产占比
    'invest_income_associates_to_total_profit',  # 联营投资收益占利润总额比
    'long_debt_to_working_capital_ratio',  # 长期负债与营运资金比
    'net_operate_cash_flow_to_total_liability',  # 经营现金流与总负债比
    'net_operating_cash_flow_coverage',    # 经营现金流覆盖倍数
    'operating_profit_to_total_profit',    # 营业利润占利润总额比
    'earnings_to_price_ratio',             # 盈利收益率（EP）
    'Kurtosis120',                         # 120日收益峰度（衡量尾部风险）
    'Kurtosis20',                          # 20日收益峰度
    'sharpe_ratio_20',                     # 20日夏普比率（风险调整收益）
    'sharpe_ratio_60',                     # 60日夏普比率
    'Skewness120',                         # 120日收益偏度（衡量分布不对称性）
    'Skewness20',                          # 20日收益偏度
    'beta',                                # Beta系数（个股相对于市场的系统性风险）
    'book_to_price_ratio',                 # 账面市值比（BP，即1/PB）
    'cash_earnings_to_price_ratio',        # 现金收益价格比
    'cube_of_size',                        # 市值的立方（用于规模因子非线性变换）
    'earnings_yield',                      # 盈利收益率
    'growth',                              # 成长性指标
]

In [ ]:
# ============================================================
# 工具函数定义
# ============================================================

def jq_symbol_to_simple(security):
    """
    把聚宽格式的股票代码转换为简洁格式。
    
    转换规则：
      '600000.XSHG' → 'sh600000'  （上海证券交易所）
      '000001.XSHE' → 'sz000001'  （深圳证券交易所）
    
    参数:
        security: 聚宽格式的股票代码字符串，如 '600000.XSHG'
    
    返回:
        简洁格式的股票代码字符串，如 'sh600000'
    """
    # 用 '.' 分割字符串，得到 [股票代码, 交易所代码]
    code, exchange = security.split('.')
    
    # XSHG = 上海证券交易所，前缀用 'sh'
    if exchange == 'XSHG':
        return 'sh' + code
    
    # XSHE = 深圳证券交易所，前缀用 'sz'
    if exchange == 'XSHE':
        return 'sz' + code
    
    # 其他情况（理论上不会出现），统一转小写返回
    return security.lower()


def build_membership_by_date(index_code=INDEX_CODE, start_date=START_DATE, end_date=END_DATE):
    """
    构建每个交易日的中证500成分股列表。
    
    为什么要做这一步？
    因为中证500指数的成分股会定期调整（每半年一次），
    今天的成分股和3年前的成分股是不一样的。
    为了获取历史因子数据，我们需要知道每个历史时间点上，
    到底是哪些股票属于中证500。
    
    参数:
        index_code: 指数代码，默认为中证500
        start_date: 起始日期
        end_date:   结束日期
    
    返回三个值:
        trade_days:        排序后的交易日日期列表
        membership_by_date: 字典，key=日期，value=该日成分股集合
        securities:        所有不重复的股票代码排序列表（历史上所有曾属于中证500的股票）
    """
    # 获取起止日期之间的所有交易日（跳过周末和法定节假日）
    # get_trade_days 是聚宽 API，返回的是 numpy 数组，转成 list 方便后续处理
    trade_days = list(get_trade_days(start_date=start_date, end_date=end_date))
    
    # membership_by_date 是一个字典：
    #   key = 日期（如 2020-01-02）
    #   value = 该日期中证500的所有成分股代码集合（如 {'600000.XSHG', '000001.XSHE', ...}）
    membership_by_date = {}
    
    # securities 用于收集历史上所有曾出现在中证500中的股票（用 set 自动去重）
    securities = set()

    # 遍历每一个交易日，i 从 1 开始编号（用于进度打印）
    for i, day in enumerate(trade_days, start=1):
        # 把 day 转为 date 对象（去掉时分秒），作为字典的 key
        day = pd.Timestamp(day).date()
        
        # 调用聚宽 API：获取该日期中证500指数的成分股列表
        # 返回的是一个列表，如 ['600000.XSHG', '000001.XSHE', ...]
        # 用 set() 转为集合，方便后续做集合运算（交集、并集等）
        day_members = set(get_index_stocks(index_code, date=day))
        
        # 记录该日的成分股
        membership_by_date[day] = day_members
        
        # 把该日成分股合并到总的 securities 集合中
        # update 会把新股票添加进去，已有的不会重复
        securities.update(day_members)

        # 每处理 250 个交易日，或者处理到最后一个交易日时，打印一次进度
        # 这样用户可以看到处理进度，不用干等
        if i % 250 == 0 or i == len(trade_days):
            print('loaded index members: {}/{} trade days'.format(i, len(trade_days)))

    # 返回三个结果：
    # 1. 按日期排序的交易日列表
    # 2. 每个交易日的成分股字典
    # 3. 排序后的所有不重复股票列表
    return sorted(membership_by_date), membership_by_date, sorted(securities)


def iter_request_windows(trade_days, membership_by_date, factors, max_cells=MAX_REQUEST_CELLS):
    """
    将整个时间范围拆分成多个"请求窗口"，确保每个窗口的请求数据量不超过 API 限制。
    
    为什么要分窗口？
    因为聚宽 API 限制单次 get_factor_values 请求的数据量不能超过 20 万个单元格。
    数据量 = 股票数 × 因子数 × 交易日数
    如果一次性请求 4696 个交易日 × 500 只股票 × 53 个因子 ≈ 1.24 亿个单元格，远超限制。
    所以需要把时间范围拆成多个小窗口，逐个请求。
    
    拆分策略：
    从起始日开始，逐步扩大窗口（增加天数），同时累计涉及的股票数，
    当数据量即将超过 max_cells 时，就切出一个窗口，然后从下一个交易日开始新窗口。
    
    参数:
        trade_days:         所有交易日列表
        membership_by_date: 每日成分股字典
        factors:            因子名称列表
        max_cells:          单次请求的最大数据单元格数（默认 180000）
    
    生成（yield）:
        每次产生一个 (window_days, window_securities) 元组：
        - window_days: 该窗口包含的交易日列表
        - window_securities: 该窗口涉及的所有股票排序列表
    """
    start = 0                # 窗口起始位置的索引
    factor_count = len(factors)  # 因子总数

    # 当起始位置还没超过交易日列表长度时，继续切分窗口
    while start < len(trade_days):
        securities = set()   # 收集当前窗口涉及的所有股票
        end = start          # 窗口结束位置，从起始位置开始逐步扩大

        # 逐步扩大窗口，看能包含多少个交易日
        while end < len(trade_days):
            day = trade_days[end]  # 当前考察的交易日
            
            # 把该日的成分股加入当前窗口的股票集合
            # 用并集（|）操作，因为不同日期的成分股可能有变化
            next_securities = securities | membership_by_date.get(day, set())
            
            # 计算如果包含到第 end 天，总的数据单元格数
            # 公式：股票数 × 因子数 × 天数
            request_cells = len(next_securities) * factor_count * (end - start + 1)

            # 如果加入这一天后数据量超限了，并且窗口内已经至少有1天，
            # 就不加入这一天，结束当前窗口
            if end > start and request_cells > max_cells:
                break

            # 如果还没超限，就把该日成分股正式加入
            securities = next_securities
            end += 1  # 继续尝试扩大窗口

            # 特殊情况：即使只包含1天也超限了，那就只能这1天单独作为一个窗口
            if request_cells > max_cells:
                break

        # 产生当前窗口的结果：包含的交易日列表 和 涉及的股票列表
        yield trade_days[start:end], sorted(securities)
        
        # 下一个窗口从当前窗口结束位置开始
        start = end


def factor_values_to_frame(factor_data):
    """
    把 get_factor_values 返回的原始数据转换为长表格式的 DataFrame。
    
    get_factor_values 返回的数据格式：
    {
        'factor_name_1': DataFrame(行=日期, 列=股票代码, 值=因子值),
        'factor_name_2': DataFrame(行=日期, 列=股票代码, 值=因子值),
        ...
    }
    这是一种"宽表"格式（每个因子一个二维表）。
    
    本函数把它转换为"长表"格式：
        date       | security     | factor_1 | factor_2 | ...
        2020-01-02 | 600000.XSHG | 1.23     | 4.56     | ...
        2020-01-02 | 000001.XSHE | 2.34     | 5.67     | ...
        ...
    
    这种格式更适合写入 CSV 文件和后续的数据分析。
    
    参数:
        factor_data: get_factor_values 返回的字典
    
    返回:
        长表格式的 DataFrame，每行是一个"日期+股票"组合的所有因子值
    """
    # 用于收集每个因子转换后的 Series
    factor_series = []

    # 遍历每个因子（factor_data 是一个字典，key=因子名，value=宽表 DataFrame）
    for factor_name, wide_df in factor_data.items():
        # 复制一份，避免修改原始数据
        wide_df = wide_df.copy()
        
        # 把行索引（日期）从字符串转为 date 对象，去掉时分秒
        wide_df.index = pd.to_datetime(wide_df.index).date
        wide_df.index.name = 'date'       # 给行索引起名
        
        # 给列索引（股票代码）起名
        wide_df.columns.name = 'security'
        
        # stack() 操作：把列（股票代码）"折叠"到行上
        # 效果：二维表变成一维 Series，每个元素是 (日期, 股票) → 因子值
        # dropna=False 表示保留 NaN 值（有些股票某些日期可能没有数据）
        # rename(factor_name) 给这个 Series 赋一个名字（因子名），后续合并时能对齐
        series = wide_df.stack(dropna=False).rename(factor_name)
        factor_series.append(series)

    # 把所有因子的 Series 横向拼接（axis=1）
    # 效果：每个 (日期, 股票) 组合作为一行，各因子的值作为不同列
    # reset_index() 把行索引（日期、股票代码）变成普通列
    return pd.concat(factor_series, axis=1).reset_index()


def progress_iter(iterable, total=None, desc=None):
    """
    包装可迭代对象，如果 tqdm 可用就显示进度条，否则原样返回。
    
    参数:
        iterable: 要遍历的对象（如列表）
        total:    总数量（用于进度条显示）
        desc:     进度条描述文字
    
    返回:
        带（或不带）进度条的可迭代对象
    """
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc)
    return iterable


def log_step(message):
    """
    打印带时间戳的日志信息，方便追踪各个步骤的执行时间。
    
    参数:
        message: 要打印的日志内容
    flush=True 确保日志立即输出到控制台，不会被缓冲区延迟
    """
    print(time.strftime('%H:%M:%S'), message, flush=True)

In [ ]:
# ============================================================
# 第一步：获取中证500所有历史成分股
# ============================================================
# 调用上面定义的 build_membership_by_date 函数，
# 遍历从 2007-01-15 到 2026-05-16 的每一个交易日，
# 查询该日中证500指数包含哪些股票。
#
# 返回三个变量：
#   trade_days         - 所有交易日列表（约4696个）
#   membership_by_date - 字典：每个交易日 → 该日成分股集合
#   securities_list    - 历史上所有曾属于中证500的不重复股票列表（约1778只）
trade_days, membership_by_date, securities_list = build_membership_by_date()

# 打印统计信息
print('trade days:', len(trade_days))         # 交易日总数
print('unique securities:', len(securities_list))  # 不重复的股票总数

# 展示前5个交易日的成分股数量，可以看到每天都是500只股票
pd.DataFrame(
    [{'date': day, 'security_count': len(membership_by_date[day])} for day in trade_days[:5]]
)

In [ ]:
# ============================================================
# 第三步：分窗口提取因子数据并写入 CSV
# ============================================================
# 整体流程：
#   1. 把整个时间范围拆成多个小窗口（因为 API 有单次请求数据量限制）
#   2. 对每个窗口调用 get_factor_values 获取因子数据
#   3. 把返回的宽表数据转换为长表格式
#   4. 过滤掉不属于该日成分股的数据（只保留当天真正属于中证500的股票）
#   5. 转换股票代码格式，按日期和股票排序，追加写入 CSV
# ============================================================

# 如果输出文件已经存在，先删除它，避免追加写入时出现旧数据残留
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

# 把所有请求窗口预先计算好，转成列表
# iter_request_windows 是一个生成器，这里用 list() 一次性获取所有窗口
# 每个窗口包含：该窗口的交易日列表 + 涉及的股票列表
windows = list(iter_request_windows(trade_days, membership_by_date, jqfactors_list))

# 打印总共有多少个窗口需要处理
log_step('request windows: {}'.format(len(windows)))

# first_write 标记是否是第一次写入文件（第一次需要写表头，后续追加不需要）
first_write = True

# 记录总共写入了多少行数据
total_rows_written = 0

# 遍历每个窗口，带进度条显示
for window_no, (window_days, window_securities) in progress_iter(
    enumerate(windows, start=1), total=len(windows), desc='factor windows'
):
    window_t0 = time.time()  # 记录当前窗口开始时间
    
    # ---- 日志：当前窗口的基本信息 ----
    # 取窗口的第一天和最后一天作为日期范围
    start_date = window_days[0].strftime('%Y-%m-%d')
    end_date = window_days[-1].strftime('%Y-%m-%d')
    
    # 计算该窗口的请求数据量 = 股票数 × 因子数 × 天数
    request_cells = len(window_securities) * len(jqfactors_list) * len(window_days)
    log_step(
        'window {}/{} start: {} ~ {}, {} days, {} securities, {} request cells'.format(
            window_no, len(windows), start_date, end_date,
            len(window_days), len(window_securities), request_cells
        )
    )

    # ---- 调用聚宽 API 获取因子数据 ----
    api_t0 = time.time()
    
    # 核心调用：get_factor_values
    # 参数说明：
    #   securities = 该窗口涉及的所有股票代码列表
    #   factors    = 要提取的53个因子名称列表
    #   start_date = 窗口起始日期
    #   end_date   = 窗口结束日期
    # 返回值是一个字典：{因子名: DataFrame(行=日期, 列=股票, 值=因子值)}
    factor_data = get_factor_values(
        securities=window_securities,
        factors=jqfactors_list,
        start_date=start_date,
        end_date=end_date,
    )
    
    log_step('window {}/{} api done: {:.1f}s'.format(window_no, len(windows), time.time() - api_t0))

    # ---- 数据格式转换：宽表 → 长表 ----
    reshape_t0 = time.time()
    
    # 调用前面定义的 factor_values_to_frame 函数
    # 把 {因子名: 宽表DataFrame} 转换为一个长表 DataFrame
    # 长表格式：每行 = (日期, 股票代码, 因子1值, 因子2值, ...)
    factor_frame = factor_values_to_frame(factor_data)
    
    log_step(
        'window {}/{} reshape done: {} rows, {:.1f}s'.format(
            window_no, len(windows), len(factor_frame), time.time() - reshape_t0
        )
    )

    # ---- 过滤：只保留当天真正属于中证500的股票 ----
    filter_t0 = time.time()
    
    # 为什么需要过滤？
    # 因为 get_factor_values 的 securities 参数是窗口内所有日期涉及的股票的并集，
    # 但在某个具体日期，有些股票可能还没进入（或已移出）中证500。
    # 我们只需要保留"该日确实属于中证500成分股"的行。
    
    # 构建该窗口内所有合法的 (日期, 股票) 组合
    window_membership = pd.MultiIndex.from_tuples(
        (day, security)
        for day in window_days
        for security in membership_by_date[day]  # 该日的成分股列表
    )
    
    # 构建当前数据帧的 (日期, 股票) 多级索引
    factor_index = pd.MultiIndex.from_tuples(
        zip(factor_frame['date'], factor_frame['security'])
    )
    
    # 用 isin 做过滤：只保留出现在 window_membership 中的行
    # 即只保留"该日确实属于中证500"的 (日期, 股票) 行
    factor_frame = factor_frame.loc[factor_index.isin(window_membership)].copy()
    
    # ---- 格式转换 ----
    # 1. 把 security 列从聚宽格式（'600000.XSHG'）转为简洁格式（'sh600000'）
    factor_frame.insert(1, 'symbol', factor_frame.pop('security').map(jq_symbol_to_simple))
    
    # 2. 把日期格式化为 'YYYY-MM-DD' 字符串（统一输出格式）
    factor_frame['date'] = pd.to_datetime(factor_frame['date']).dt.strftime('%Y-%m-%d')
    
    # 3. 重新排列列的顺序：日期、股票代码、然后是53个因子列
    factor_frame = factor_frame[['date', 'symbol'] + jqfactors_list]
    
    # 4. 按日期升序、同一日内按股票代码升序排列
    factor_frame = factor_frame.sort_values(['date', 'symbol'])
    
    log_step(
        'window {}/{} filter/sort done: {} rows, {:.1f}s'.format(
            window_no, len(windows), len(factor_frame), time.time() - filter_t0
        )
    )

    # ---- 写入 CSV 文件 ----
    write_t0 = time.time()
    
    # 写入参数说明：
    #   mode='w' (第一次) / 'a' (追加)  — 第一次写表头，后续追加数据
    #   header=first_write               — 第一次写列名，后续不写
    #   index=False                      — 不写入 DataFrame 的行索引
    factor_frame.to_csv(
        OUTPUT_CSV,
        mode='w' if first_write else 'a',
        header=first_write,
        index=False,
    )
    
    rows_written = len(factor_frame)    # 本窗口写入的行数
    total_rows_written += rows_written  # 累计总行数
    first_write = False                 # 第一次之后都改为追加模式
    
    # ---- 内存清理 ----
    # 删除本次窗口的中间变量，释放内存
    # 因为处理大量数据时，如果不及时释放，内存会持续增长导致程序崩溃
    del factor_data, factor_frame, factor_index, window_membership
    gc.collect()  # 手动触发垃圾回收，确保内存被释放
    
    log_step(
        'window {}/{} wrote {} rows, total {} rows, write {:.1f}s, window {:.1f}s'.format(
            window_no, len(windows), rows_written, total_rows_written,
            time.time() - write_t0, time.time() - window_t0
        )
    )

# 全部处理完成
log_step('done: {}'.format(OUTPUT_CSV))

In [ ]:
# ============================================================
# 第四步：预览输出结果
# ============================================================
# 读取 CSV 文件的前5行，检查数据格式是否正确：
#   - date 列：日期，格式为 YYYY-MM-DD
#   - symbol 列：股票代码，格式为 sh600000 / sz000001
#   - 后续53列：各因子的数值
preview = pd.read_csv(OUTPUT_CSV, nrows=5)

# 打印数据形状：(行数, 列数)，列数 = 2(日期+股票) + 53(因子) = 55
print(preview.shape)

# 显示前5行数据
preview